In [1]:
# ============================================================
# Cell 1: 导入 & 加载数据
# ============================================================
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import joblib
from sklearn.metrics import r2_score

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('使用设备:', DEVICE)

# 加载scaler（UK）
scaler_X = joblib.load('/root/scaler_X.pkl')
scaler_y = joblib.load('/root/scaler_y.pkl')

# 加载数据
X_uk = np.load('/root/X_uk.npy', mmap_mode='r')
y_uk = np.load('/root/y_uk.npy', mmap_mode='r')
X_ie = np.load('/root/X_ie.npy', mmap_mode='r')
y_ie = np.load('/root/y_ie.npy', mmap_mode='r')

# 时间索引
N_TIMES    = 17532
N_GRID_UK  = 2793
N_GRID_IE  = 525

uk_times = pd.date_range('2020-01-01 13:00', periods=N_TIMES, freq='h')
uk_train_mask = uk_times < '2021-01-01'
uk_val_mask   = (uk_times >= '2021-01-01') & (uk_times < '2021-07-01')
uk_test_mask  = uk_times >= '2021-07-01'

ie_times = pd.date_range('2020-01-01 13:00', periods=N_TIMES, freq='h')
ie_test_mask = ie_times >= '2021-07-01'

train_idx    = np.where(np.tile(uk_train_mask, N_GRID_UK))[0]
val_idx      = np.where(np.tile(uk_val_mask,   N_GRID_UK))[0]
test_idx_uk  = np.where(np.tile(uk_test_mask,  N_GRID_UK))[0]
test_idx_ie  = np.where(np.tile(ie_test_mask,  N_GRID_IE))[0]

print(f'UK  Train: {len(train_idx):,} | Val: {len(val_idx):,} | Test: {len(test_idx_uk):,}')
print(f'IE  Test:  {len(test_idx_ie):,}')
print('数据加载完成')

使用设备: cuda
UK  Train: 24,497,403 | Val: 12,132,792 | Test: 12,336,681
IE  Test:  2,318,925
数据加载完成


In [2]:
# ============================================================
# Cell 2: Dataset & DataLoader
# ============================================================
class RainfallDataset(Dataset):
    def __init__(self, X, y, idx, scaler_X, scaler_y):
        X_data = X[idx].reshape(-1, 7)
        X_data = scaler_X.transform(X_data).reshape(-1, 12, 7)
        self.X = torch.tensor(X_data, dtype=torch.float32)
        y_data = y[idx].reshape(-1, 1)
        y_data = scaler_y.transform(y_data).reshape(-1)
        self.y = torch.tensor(y_data, dtype=torch.float32)
    def __len__(self):
        return len(self.X)
    def __getitem__(self, i):
        return self.X[i], self.y[i]

print('创建 UK DataLoader...')
train_dataset = RainfallDataset(X_uk, y_uk, train_idx,   scaler_X, scaler_y)
val_dataset   = RainfallDataset(X_uk, y_uk, val_idx,     scaler_X, scaler_y)
test_dataset  = RainfallDataset(X_uk, y_uk, test_idx_uk, scaler_X, scaler_y)

train_loader = DataLoader(train_dataset, batch_size=4096, shuffle=True,  num_workers=4)
val_loader   = DataLoader(val_dataset,   batch_size=4096, shuffle=False, num_workers=4)
test_loader  = DataLoader(test_dataset,  batch_size=4096, shuffle=False, num_workers=4)

print('创建 IE test DataLoader (用UK scaler，zero-shot)...')
ie_test_dataset = RainfallDataset(X_ie, y_ie, test_idx_ie, scaler_X, scaler_y)
ie_test_loader  = DataLoader(ie_test_dataset, batch_size=4096, shuffle=False, num_workers=4)

print('DataLoader 创建完成')

创建 UK DataLoader...
创建 IE test DataLoader (用UK scaler，zero-shot)...
DataLoader 创建完成


In [3]:
# ============================================================
# Cell 3: 共用函数
# ============================================================
def pinball_loss(pred, target, quantile):
    err = target - pred
    return torch.where(err >= 0, quantile * err, (quantile - 1) * err).mean()

def evaluate(model, loader, scaler_y, device, model_type='pg'):
    """
    通用评估函数
    model_type: 'pg' 表示返回gate的模型，'nogate' 表示不返回gate
    返回: RMSE, R², Coverage, CSI, FAR
    """
    model.eval()
    all_q10, all_q50, all_q90, all_true = [], [], [], []
    with torch.no_grad():
        for X_batch, y_batch in loader:
            X_batch = X_batch.to(device)
            if model_type == 'pg':
                q10, q50, q90, _ = model(X_batch)
            else:
                q10, q50, q90 = model(X_batch)
            all_q10.append(q10.cpu().numpy())
            all_q50.append(q50.cpu().numpy())
            all_q90.append(q90.cpu().numpy())
            all_true.append(y_batch.numpy())

    q10s  = np.concatenate(all_q10)
    q50s  = np.concatenate(all_q50)
    q90s  = np.concatenate(all_q90)
    trues = np.concatenate(all_true)

    # 反标准化
    q10_orig   = scaler_y.inverse_transform(q10s.reshape(-1,1)).reshape(-1)
    q50_orig   = scaler_y.inverse_transform(q50s.reshape(-1,1)).reshape(-1)
    q90_orig   = scaler_y.inverse_transform(q90s.reshape(-1,1)).reshape(-1)
    trues_orig = scaler_y.inverse_transform(trues.reshape(-1,1)).reshape(-1)

    # RMSE & R²
    rmse     = np.sqrt(np.mean((q50_orig - trues_orig)**2))
    r2       = r2_score(trues_orig, q50_orig)

    # Coverage
    coverage = np.mean((trues_orig >= q10_orig) & (trues_orig <= q90_orig))

    # CSI & FAR（用q90预测强降雨，95th percentile阈值）
    threshold = np.percentile(trues_orig, 95)
    obs_event  = trues_orig > threshold
    pred_event = q90_orig   > threshold
    H = np.sum( obs_event &  pred_event)
    M = np.sum( obs_event & ~pred_event)
    F = np.sum(~obs_event &  pred_event)
    csi = H / (H + M + F) if (H + M + F) > 0 else 0.0
    far = F / (H + F)     if (H + F)     > 0 else 0.0

    return rmse, r2, coverage, csi, far

def print_results(name, rmse, r2, coverage, csi, far):
    print(f'=== {name} ===')
    print(f'  RMSE:     {rmse:.6f}')
    print(f'  R²:       {r2:.4f}')
    print(f'  Coverage: {coverage:.4f}  (理想值=0.80)')
    print(f'  CSI:      {csi:.4f}')
    print(f'  FAR:      {far:.4f}')
    print()

print('共用函数定义完成')

共用函数定义完成


---
## 变体1：No-Gate（去掉physics gate，保留extreme loss）

In [4]:
# ============================================================
# Cell 4: No-Gate 模型定义
# ============================================================
class NoGateQuantileLSTM(nn.Module):
    """去掉physics gate，其余结构与PhysicsGuidedQuantileLSTM完全相同"""
    def __init__(self, input_size=7, hidden_size=64, num_layers=2):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers,
                            batch_first=True, dropout=0.2)
        self.fc_q10 = nn.Linear(hidden_size, 1)
        self.fc_q50 = nn.Linear(hidden_size, 1)
        self.fc_q90 = nn.Linear(hidden_size, 1)

    def forward(self, x):
        out, _ = self.lstm(x)
        h = out[:, -1, :]
        # 直接输出，不经过gate
        q10 = self.fc_q10(h).squeeze(-1)
        q50 = self.fc_q50(h).squeeze(-1)
        q90 = self.fc_q90(h).squeeze(-1)
        return q10, q50, q90

def nogate_loss(q10, q50, q90, target, extreme_threshold=0.95):
    """保留extreme loss，去掉gate（loss与完整版相同，只是模型结构不同）"""
    base_loss = (pinball_loss(q10, target, 0.1) +
                 pinball_loss(q50, target, 0.5) +
                 pinball_loss(q90, target, 0.9))
    threshold    = torch.quantile(target, extreme_threshold)
    extreme_mask = (target > threshold).float()
    extreme_weight = 1.0 + 2.0 * extreme_mask
    weighted_loss = (pinball_loss(q10, target, 0.1) * extreme_weight +
                     pinball_loss(q50, target, 0.5) * extreme_weight +
                     pinball_loss(q90, target, 0.9) * extreme_weight).mean()
    return base_loss + 0.5 * weighted_loss

print('No-Gate 模型定义完成')

No-Gate 模型定义完成


In [5]:
# ============================================================
# Cell 5: No-Gate 训练
# ============================================================
model_nogate = NoGateQuantileLSTM().to(DEVICE)
optimizer_nogate = torch.optim.Adam(model_nogate.parameters(), lr=3e-4)

EPOCHS  = 30
PATIENCE = 5
best_val_loss    = float('inf')
patience_counter = 0

print('=== No-Gate 训练开始 ===')
for epoch in range(EPOCHS):
    model_nogate.train()
    train_loss = 0
    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(DEVICE), y_batch.to(DEVICE)
        optimizer_nogate.zero_grad()
        q10, q50, q90 = model_nogate(X_batch)
        loss = nogate_loss(q10, q50, q90, y_batch)
        loss.backward()
        optimizer_nogate.step()
        train_loss += loss.item()
    train_loss /= len(train_loader)

    model_nogate.eval()
    val_loss = 0
    with torch.no_grad():
        for X_batch, y_batch in val_loader:
            X_batch, y_batch = X_batch.to(DEVICE), y_batch.to(DEVICE)
            q10, q50, q90 = model_nogate(X_batch)
            val_loss += nogate_loss(q10, q50, q90, y_batch).item()
    val_loss /= len(val_loader)

    print(f'Epoch {epoch+1}/{EPOCHS} | Train: {train_loss:.4f} | Val: {val_loss:.4f}')
    if val_loss < best_val_loss:
        best_val_loss    = val_loss
        patience_counter = 0
        torch.save(model_nogate.state_dict(), '/root/best_ablation_nogate_t1.pth')
        print(f'  ✓ 保存最优模型 (val_loss={val_loss:.4f})')
    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            print(f'早停！连续{PATIENCE}个epoch没有改善')
            break

print(f'No-Gate 训练完成！最优Val Loss: {best_val_loss:.4f}')

=== No-Gate 训练开始 ===
Epoch 1/30 | Train: 0.3068 | Val: 0.2185
  ✓ 保存最优模型 (val_loss=0.2185)
Epoch 2/30 | Train: 0.2780 | Val: 0.2164
  ✓ 保存最优模型 (val_loss=0.2164)
Epoch 3/30 | Train: 0.2745 | Val: 0.2172
Epoch 4/30 | Train: 0.2722 | Val: 0.2164
  ✓ 保存最优模型 (val_loss=0.2164)
Epoch 5/30 | Train: 0.2703 | Val: 0.2166
Epoch 6/30 | Train: 0.2687 | Val: 0.2153
  ✓ 保存最优模型 (val_loss=0.2153)
Epoch 7/30 | Train: 0.2673 | Val: 0.2157
Epoch 8/30 | Train: 0.2661 | Val: 0.2154
Epoch 9/30 | Train: 0.2650 | Val: 0.2163
Epoch 10/30 | Train: 0.2640 | Val: 0.2157
Epoch 11/30 | Train: 0.2631 | Val: 0.2162
早停！连续5个epoch没有改善
No-Gate 训练完成！最优Val Loss: 0.2153


In [6]:
# ============================================================
# Cell 6: No-Gate 评估（UK test + IE zero-shot）
# ============================================================
model_nogate.load_state_dict(torch.load('/root/best_ablation_nogate_t1.pth'))

rmse, r2, cov, csi, far = evaluate(model_nogate, test_loader,    scaler_y, DEVICE, model_type='nogate')
print_results('No-Gate | UK Test (t+1)', rmse, r2, cov, csi, far)

rmse, r2, cov, csi, far = evaluate(model_nogate, ie_test_loader, scaler_y, DEVICE, model_type='nogate')
print_results('No-Gate | IE Zero-shot (t+1)', rmse, r2, cov, csi, far)

=== No-Gate | UK Test (t+1) ===
  RMSE:     0.000196
  R²:       0.7079
  Coverage: 0.8363  (理想值=0.80)
  CSI:      0.4529
  FAR:      0.5194

=== No-Gate | IE Zero-shot (t+1) ===
  RMSE:     0.000210
  R²:       0.6948
  Coverage: 0.8336  (理想值=0.80)
  CSI:      0.4411
  FAR:      0.5291



---
## 变体2：No-Extreme（保留physics gate，去掉extreme loss，lambda=0）

In [7]:
# ============================================================
# Cell 7: No-Extreme 模型定义
# ============================================================
class NoExtremeQuantileLSTM(nn.Module):
    """保留physics gate，loss只用基础pinball（lambda=0）"""
    def __init__(self, input_size=7, hidden_size=64, num_layers=2):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers,
                            batch_first=True, dropout=0.2)
        self.gate_fc = nn.Sequential(
            nn.Linear(2, 16),
            nn.ReLU(),
            nn.Linear(16, 1),
            nn.Sigmoid()
        )
        self.fc_q10 = nn.Linear(hidden_size, 1)
        self.fc_q50 = nn.Linear(hidden_size, 1)
        self.fc_q90 = nn.Linear(hidden_size, 1)

    def forward(self, x):
        out, _ = self.lstm(x)
        h = out[:, -1, :]
        # physics gate（sp=index4, tcc=index5）
        tcc_sp = x[:, -1, 4:6]
        gate   = self.gate_fc(tcc_sp)
        q10 = (self.fc_q10(h) * gate).squeeze(-1)
        q50 = (self.fc_q50(h) * gate).squeeze(-1)
        q90 = (self.fc_q90(h) * gate).squeeze(-1)
        return q10, q50, q90, gate.squeeze(-1)

def noextreme_loss(q10, q50, q90, target):
    """只用基础pinball loss，不加extreme weighting（lambda=0）"""
    return (pinball_loss(q10, target, 0.1) +
            pinball_loss(q50, target, 0.5) +
            pinball_loss(q90, target, 0.9))

print('No-Extreme 模型定义完成')

No-Extreme 模型定义完成


In [8]:
# ============================================================
# Cell 8: No-Extreme 训练
# ============================================================
model_noext = NoExtremeQuantileLSTM().to(DEVICE)
optimizer_noext = torch.optim.Adam(model_noext.parameters(), lr=3e-4)

EPOCHS  = 30
PATIENCE = 5
best_val_loss    = float('inf')
patience_counter = 0

print('=== No-Extreme 训练开始 ===')
for epoch in range(EPOCHS):
    model_noext.train()
    train_loss = 0
    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(DEVICE), y_batch.to(DEVICE)
        optimizer_noext.zero_grad()
        q10, q50, q90, gate = model_noext(X_batch)
        loss = noextreme_loss(q10, q50, q90, y_batch)
        loss.backward()
        optimizer_noext.step()
        train_loss += loss.item()
    train_loss /= len(train_loader)

    model_noext.eval()
    val_loss = 0
    with torch.no_grad():
        for X_batch, y_batch in val_loader:
            X_batch, y_batch = X_batch.to(DEVICE), y_batch.to(DEVICE)
            q10, q50, q90, gate = model_noext(X_batch)
            val_loss += noextreme_loss(q10, q50, q90, y_batch).item()
    val_loss /= len(val_loader)

    print(f'Epoch {epoch+1}/{EPOCHS} | Train: {train_loss:.4f} | Val: {val_loss:.4f}')
    if val_loss < best_val_loss:
        best_val_loss    = val_loss
        patience_counter = 0
        torch.save(model_noext.state_dict(), '/root/best_ablation_noextreme_t1.pth')
        print(f'  ✓ 保存最优模型 (val_loss={val_loss:.4f})')
    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            print(f'早停！连续{PATIENCE}个epoch没有改善')
            break

print(f'No-Extreme 训练完成！最优Val Loss: {best_val_loss:.4f}')

=== No-Extreme 训练开始 ===
Epoch 1/30 | Train: 0.1983 | Val: 0.1409
  ✓ 保存最优模型 (val_loss=0.1409)
Epoch 2/30 | Train: 0.1788 | Val: 0.1393
  ✓ 保存最优模型 (val_loss=0.1393)
Epoch 3/30 | Train: 0.1764 | Val: 0.1398
Epoch 4/30 | Train: 0.1749 | Val: 0.1384
  ✓ 保存最优模型 (val_loss=0.1384)
Epoch 5/30 | Train: 0.1737 | Val: 0.1387
Epoch 6/30 | Train: 0.1727 | Val: 0.1384
  ✓ 保存最优模型 (val_loss=0.1384)
Epoch 7/30 | Train: 0.1719 | Val: 0.1386
Epoch 8/30 | Train: 0.1711 | Val: 0.1389
Epoch 9/30 | Train: 0.1704 | Val: 0.1387
Epoch 10/30 | Train: 0.1698 | Val: 0.1395
Epoch 11/30 | Train: 0.1692 | Val: 0.1392
早停！连续5个epoch没有改善
No-Extreme 训练完成！最优Val Loss: 0.1384


In [9]:
# ============================================================
# Cell 9: No-Extreme 评估（UK test + IE zero-shot）
# ============================================================
model_noext.load_state_dict(torch.load('/root/best_ablation_noextreme_t1.pth'))

rmse, r2, cov, csi, far = evaluate(model_noext, test_loader,    scaler_y, DEVICE, model_type='pg')
print_results('No-Extreme | UK Test (t+1)', rmse, r2, cov, csi, far)

rmse, r2, cov, csi, far = evaluate(model_noext, ie_test_loader, scaler_y, DEVICE, model_type='pg')
print_results('No-Extreme | IE Zero-shot (t+1)', rmse, r2, cov, csi, far)

=== No-Extreme | UK Test (t+1) ===
  RMSE:     0.000194
  R²:       0.7140
  Coverage: 0.8457  (理想值=0.80)
  CSI:      0.4512
  FAR:      0.5219

=== No-Extreme | IE Zero-shot (t+1) ===
  RMSE:     0.000208
  R²:       0.7002
  Coverage: 0.8430  (理想值=0.80)
  CSI:      0.4395
  FAR:      0.5316



---
## 汇总结果（复制到论文表格）

In [10]:
# ============================================================
# Cell 10: 汇总对比表
# ============================================================
print('重新加载两个变体模型并汇总结果...')

model_nogate.load_state_dict(torch.load('/root/best_ablation_nogate_t1.pth'))
model_noext.load_state_dict(torch.load('/root/best_ablation_noextreme_t1.pth'))

results = {}

# No-Gate
r = evaluate(model_nogate, test_loader,    scaler_y, DEVICE, 'nogate')
results['No-Gate | UK']       = r
r = evaluate(model_nogate, ie_test_loader, scaler_y, DEVICE, 'nogate')
results['No-Gate | IE']       = r

# No-Extreme
r = evaluate(model_noext, test_loader,    scaler_y, DEVICE, 'pg')
results['No-Extreme | UK']    = r
r = evaluate(model_noext, ie_test_loader, scaler_y, DEVICE, 'pg')
results['No-Extreme | IE']    = r

print()
print(f'{"Model":<25} {"RMSE":>10} {"R²":>8} {"Coverage":>10} {"CSI":>8} {"FAR":>8}')
print('-' * 75)
for name, (rmse, r2, cov, csi, far) in results.items():
    print(f'{name:<25} {rmse:>10.6f} {r2:>8.4f} {cov:>10.4f} {csi:>8.4f} {far:>8.4f}')
print()
print('完成！将以上数字填入论文消融实验表格。')

重新加载两个变体模型并汇总结果...

Model                           RMSE       R²   Coverage      CSI      FAR
---------------------------------------------------------------------------
No-Gate | UK                0.000196   0.7079     0.8363   0.4529   0.5194
No-Gate | IE                0.000210   0.6948     0.8336   0.4411   0.5291
No-Extreme | UK             0.000194   0.7140     0.8457   0.4512   0.5219
No-Extreme | IE             0.000208   0.7002     0.8430   0.4395   0.5316

完成！将以上数字填入论文消融实验表格。
